# 07 — Caliper × video-log × Ardaman panel

Two-panel composite for each priority well:

- **Left**: caliper trace + per-sample severity bands. Companion-well
  caliper traces (suffix O for the 2009 well, S for the shallow
  companion) are overlaid for visual cross-check.
- **Right**: video-log notes and (when applicable) Ardaman 2009
  lithology + in-situ-conductivity entries, anchored to their depths
  via brackets and leader lines.

## Convention

Y-axis is "Depth below ground level (m)" with positive values
(0 = ground level at top, deeper toward the bottom). Matches SEC
and the rest of the project. "Elevation" stays reserved for absolute
elevation (m above sea level) once differential GPS data is in.

## Pipeline

1. Inputs: per-sample CSV (from ``caliper_run_pipeline.py``),
   master caliper CSV, video xlsx, Ardaman csv.
2. Loaders parse the xlsx and Ardaman csv into uniform DataFrames
   with ``depth_*_bgl_m`` columns.
3. Layout uses a PAV-isotonic minimum-displacement algorithm to
   keep label centres close to their data anchors without overlapping.
4. One PNG per well is written.

## Note on cross-borehole correlation

The video well is NOT always the same as the calipered well.
See ``WELLS`` in ``karst_analysis.convergence`` for the mapping.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd
import matplotlib.pyplot as plt

from karst_analysis.videolog import load_video_notes
from karst_analysis.drilling import load_ardaman
from karst_analysis.convergence import (
    WELLS, plot_caliper_video_panel, build_all_caliper_video_panels,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print("Wells available:")
for w, cfg in WELLS.items():
    print(f"  {w:8s}  video sheet={cfg.video_sheet:8s}  has_ardaman={cfg.has_ardaman}")

## 1. Inspect a video-log sheet

In [ ]:
notes = load_video_notes(sheet="LRS70D")
print(f"LRS70D: {len(notes)} entries")
print(f"depth_centre_bgl_m range: [{notes['depth_centre_bgl_m'].min():.2f}, {notes['depth_centre_bgl_m'].max():.2f}]")
notes.head(8)

## 2. Inspect Ardaman entries (only AW5O / AW6O)

In [ ]:
ard = load_ardaman(well="AW5O")
print(f"AW5O: {len(ard)} Ardaman entries")
ard.head(10)

## 3. Render the panel for a single well

In [ ]:
fig = plot_caliper_video_panel(
    "LRS70D",
    output_path="results/figures/convergence/caliper_video/LRS70D_caliper_videolog_panel.png",
)
plt.close(fig)
print("✓ Saved")

from IPython.display import Image
Image(filename="results/figures/convergence/caliper_video/LRS70D_caliper_videolog_panel.png")

## 4. Render all five panels at once

In [ ]:
paths = build_all_caliper_video_panels(
    output_dir="results/figures/convergence/caliper_video",
)
print(f"\n{len(paths)} panels written.")